In [7]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("database.sqlite")

def q(sql):
    return pd.read_sql_query(sql, conn)

In [8]:
q('''
SELECT name
FROM sqlite_master
WHERE type = 'table';

''')

,name
0,Player
1,Extra_Runs
2,Batsman_Scored
3,Batting_Style
4,Bowling_Style
5,Country
6,Season
7,City
8,Outcome
9,Win_By


In [10]:
for table_name in tables["name"]:
    df = pd.read_sql_query(f"SELECT * FROM {table_name} LIMIT 5;", conn)

    print("=" * 50)
    print("Table:", table_name)
    print("Columns:")
    print(list(df.columns))

Table: Player
Columns:
['Player_Id', 'Player_Name', 'DOB', 'Batting_hand', 'Bowling_skill', 'Country_Name']
Table: Extra_Runs
Columns:
['Match_Id', 'Over_Id', 'Ball_Id', 'Extra_Type_Id', 'Extra_Runs', 'Innings_No']
Table: Batsman_Scored
Columns:
['Match_Id', 'Over_Id', 'Ball_Id', 'Runs_Scored', 'Innings_No']
Table: Batting_Style
Columns:
['Batting_Id', 'Batting_hand']
Table: Bowling_Style
Columns:
['Bowling_Id', 'Bowling_skill']
Table: Country
Columns:
['Country_Id', 'Country_Name']
Table: Season
Columns:
['Season_Id', 'Man_of_the_Series', 'Orange_Cap', 'Purple_Cap', 'Season_Year']
Table: City
Columns:
['City_Id', 'City_Name', 'Country_id']
Table: Outcome
Columns:
['Outcome_Id', 'Outcome_Type']
Table: Win_By
Columns:
['Win_Id', 'Win_Type']
Table: Wicket_Taken
Columns:
['Match_Id', 'Over_Id', 'Ball_Id', 'Player_Out', 'Kind_Out', 'Fielders', 'Innings_No']
Table: Venue
Columns:
['Venue_Id', 'Venue_Name', 'City_Id']
Table: Extra_Type
Columns:
['Extra_Id', 'Extra_Name']
Table: Out_Type
Colu

In [9]:
# Query 1: Select All Columns from Player’s Table

# Write and execute a SQL query to select all columns from the Player_Match table.

q('''
SELECT *
from player
''')


,Player_Id,Player_Name,DOB,Batting_hand,Bowling_skill,Country_Name
0,1,SC Ganguly,1972-07-08 00:00:00,1,1.0,1
1,2,BB McCullum,1981-09-27 00:00:00,2,1.0,4
2,3,RT Ponting,1974-12-19 00:00:00,2,1.0,5
3,4,DJ Hussey,1977-07-15 00:00:00,2,2.0,5
4,5,Mohammad Hafeez,1980-10-17 00:00:00,2,2.0,6
...,...,...,...,...,...,...
464,465,DL Chahar,1992-08-07 00:00:00,2,1.0,1
465,466,P Dharmani,1974-09-27 00:00:00,2,NaN,1
466,467,RV Pawar,1979-09-06 00:00:00,1,7.0,1
467,468,KH Devdhar,1989-12-14 00:00:00,2,NaN,1


In [13]:
# Query 2: Batsman vs Runs

# Write and execute a SQL query to calculate the total runs scored by each batsman.


q("""
SELECT
    p.Player_Name AS batsman,
    SUM(bs.Runs_Scored) AS total_runs
FROM Batsman_Scored bs
JOIN Ball_by_Ball b
    ON bs.Match_Id = b.Match_Id
   AND bs.Over_Id = b.Over_Id
   AND bs.Ball_Id = b.Ball_Id
   AND bs.Innings_No = b.Innings_No
JOIN Player p
    ON b.Striker = p.Player_Id
GROUP BY p.Player_Id, p.Player_Name
ORDER BY total_runs DESC;
""")

,batsman,total_runs
0,SK Raina,4106
1,V Kohli,4105
2,RG Sharma,3874
3,G Gambhir,3634
4,CH Gayle,3447
...,...,...
429,PJ Cummins,0
430,M Ashwin,0
431,Swapnil Singh,0
432,A Zampa,0


In [14]:
# Query 3: Fifties and Hundreds

# Write and execute a SQL query to calculate the number of fifties and hundreds scored by each batsman.

q("""
WITH innings_runs AS (
    SELECT
        p.Player_Id,
        p.Player_Name AS batsman,
        bs.Match_Id,
        bs.Innings_No,
        SUM(bs.Runs_Scored) AS runs_in_innings
    FROM Batsman_Scored bs
    JOIN Ball_by_Ball b
        ON bs.Match_Id = b.Match_Id
       AND bs.Over_Id = b.Over_Id
       AND bs.Ball_Id = b.Ball_Id
       AND bs.Innings_No = b.Innings_No
    JOIN Player p
        ON b.Striker = p.Player_Id
    GROUP BY
        p.Player_Id,
        p.Player_Name,
        bs.Match_Id,
        bs.Innings_No
)

SELECT
    batsman,
    SUM(CASE
            WHEN runs_in_innings >= 50 AND runs_in_innings < 100
            THEN 1 ELSE 0
        END) AS fifties,
    SUM(CASE
            WHEN runs_in_innings >= 100
            THEN 1 ELSE 0
        END) AS hundreds
FROM innings_runs
GROUP BY Player_Id, batsman
ORDER BY hundreds DESC, fifties DESC;
""")

,batsman,fifties,hundreds
0,CH Gayle,20,5
1,V Kohli,26,4
2,AB de Villiers,21,3
3,DA Warner,32,2
4,V Sehwag,16,2
...,...,...,...
429,P Kumar,0,0
430,AA Noffke,0,0
431,B Akhil,0,0
432,Mohammad Hafeez,0,0


In [15]:
# Query 4: Best Bowling Figures

# Write and execute a SQL query to find the best bowling figures for each bowler.

q("""
WITH bowler_wickets AS (
    SELECT
        p.Player_Id,
        p.Player_Name AS bowler,
        b.Match_Id,
        b.Innings_No,
        COUNT(*) AS wickets
    FROM Wicket_Taken wt
    JOIN Ball_by_Ball b
        ON wt.Match_Id = b.Match_Id
       AND wt.Over_Id = b.Over_Id
       AND wt.Ball_Id = b.Ball_Id
       AND wt.Innings_No = b.Innings_No
    JOIN Player p
        ON b.Bowler = p.Player_Id
    JOIN Out_Type ot
        ON wt.Kind_Out = ot.Out_Id
    WHERE ot.Out_Name NOT IN ('run out', 'retired hurt', 'obstructing the field')
    GROUP BY
        p.Player_Id,
        p.Player_Name,
        b.Match_Id,
        b.Innings_No
)

SELECT
    bowler,
    MAX(wickets) AS best_bowling_wickets
FROM bowler_wickets
GROUP BY Player_Id, bowler
ORDER BY best_bowling_wickets DESC;
""")


,bowler,best_bowling_wickets
0,Sohail Tanvir,6
1,A Zampa,6
2,RA Jadeja,5
3,Harbhajan Singh,5
4,I Sharma,5
...,...,...
280,S Badree,1
281,P Suyal,1
282,GS Sandhu,1
283,AF Milne,1


In [16]:
# Query 5: Comprehensive Career Metrics

# Combine all the previous chunks into a single comprehensive query to get detailed career metrics for players.


q("""
WITH batting_runs AS (
    SELECT
        p.Player_Id,
        p.Player_Name,
        SUM(bs.Runs_Scored) AS total_runs
    FROM Batsman_Scored bs
    JOIN Ball_by_Ball b
        ON bs.Match_Id = b.Match_Id
       AND bs.Over_Id = b.Over_Id
       AND bs.Ball_Id = b.Ball_Id
       AND bs.Innings_No = b.Innings_No
    JOIN Player p
        ON b.Striker = p.Player_Id
    GROUP BY p.Player_Id, p.Player_Name
),

innings_runs AS (
    SELECT
        p.Player_Id,
        p.Player_Name,
        bs.Match_Id,
        bs.Innings_No,
        SUM(bs.Runs_Scored) AS runs_in_innings
    FROM Batsman_Scored bs
    JOIN Ball_by_Ball b
        ON bs.Match_Id = b.Match_Id
       AND bs.Over_Id = b.Over_Id
       AND bs.Ball_Id = b.Ball_Id
       AND bs.Innings_No = b.Innings_No
    JOIN Player p
        ON b.Striker = p.Player_Id
    GROUP BY
        p.Player_Id,
        p.Player_Name,
        bs.Match_Id,
        bs.Innings_No
),

fifties_hundreds AS (
    SELECT
        Player_Id,
        Player_Name,
        SUM(CASE
                WHEN runs_in_innings >= 50 AND runs_in_innings < 100
                THEN 1 ELSE 0
            END) AS fifties,
        SUM(CASE
                WHEN runs_in_innings >= 100
                THEN 1 ELSE 0
            END) AS hundreds
    FROM innings_runs
    GROUP BY Player_Id, Player_Name
),

bowling_wickets AS (
    SELECT
        p.Player_Id,
        p.Player_Name,
        b.Match_Id,
        b.Innings_No,
        COUNT(*) AS wickets_in_innings
    FROM Wicket_Taken wt
    JOIN Ball_by_Ball b
        ON wt.Match_Id = b.Match_Id
       AND wt.Over_Id = b.Over_Id
       AND wt.Ball_Id = b.Ball_Id
       AND wt.Innings_No = b.Innings_No
    JOIN Player p
        ON b.Bowler = p.Player_Id
    JOIN Out_Type ot
        ON wt.Kind_Out = ot.Out_Id
    WHERE ot.Out_Name NOT IN ('run out', 'retired hurt', 'obstructing the field')
    GROUP BY
        p.Player_Id,
        p.Player_Name,
        b.Match_Id,
        b.Innings_No
),

bowling_summary AS (
    SELECT
        Player_Id,
        Player_Name,
        SUM(wickets_in_innings) AS total_wickets,
        MAX(wickets_in_innings) AS best_bowling_wickets
    FROM bowling_wickets
    GROUP BY Player_Id, Player_Name
)

SELECT
    p.Player_Name,
    COALESCE(br.total_runs, 0) AS total_runs,
    COALESCE(fh.fifties, 0) AS fifties,
    COALESCE(fh.hundreds, 0) AS hundreds,
    COALESCE(bs.total_wickets, 0) AS total_wickets,
    COALESCE(bs.best_bowling_wickets, 0) AS best_bowling_wickets
FROM Player p
LEFT JOIN batting_runs br
    ON p.Player_Id = br.Player_Id
LEFT JOIN fifties_hundreds fh
    ON p.Player_Id = fh.Player_Id
LEFT JOIN bowling_summary bs
    ON p.Player_Id = bs.Player_Id
ORDER BY total_runs DESC, total_wickets DESC;
""")

,Player_Name,total_runs,fifties,hundreds,total_wickets,best_bowling_wickets
0,SK Raina,4106,28,1,24,2
1,V Kohli,4105,26,4,4,2
2,RG Sharma,3874,29,1,15,4
3,G Gambhir,3634,31,0,0,0
4,CH Gayle,3447,20,5,18,3
...,...,...,...,...,...,...
464,DL Chahar,0,0,0,0,0
465,P Dharmani,0,0,0,0,0
466,RV Pawar,0,0,0,0,0
467,KH Devdhar,0,0,0,0,0
